# 텍스트마이닝 ex02 · 네이버 쇼핑 리뷰 감성분석 (Word2Vec · 유사도 · 내적값)

- **주제** : 네이버 쇼핑 리뷰를 **긍정 / 부정**으로 분류하고, 단어 하나하나의 **감성 점수**까지 뽑아보기
- **데이터** : 네이버 쇼핑 리뷰 20만 건 (`data/naver_shopping.txt`, 평점 + 리뷰 2컬럼)
- **ex01과 달라진 점** : 특징값 추출을 **TF-IDF → Word2Vec 임베딩**으로 교체

| 구분 | ex01 (혐오표현) | ex02 (쇼핑 리뷰) |
|---|---|---|
| 특징값 추출 | TF-IDF (단어 빈도 기반) | **Word2Vec 평균 벡터** (의미 기반) |
| 벡터 차원 | 단어 개수만큼 (수만 차원, 희소) | 100차원 (조밀) |
| 단어 의미 | 반영 안 됨 | **반영됨** (유사도 계산 가능) |
| 평가 | train/test 분리 | **교차검증 (StratifiedKFold 5겹)** |

### 파이프라인
데이터 수집 → 정제(정규표현식) → 토큰화(Kiwi + Okt) → **Word2Vec 임베딩** → 문서 평균 벡터 → 로지스틱 회귀 → **내적으로 단어 감성 점수**

### 목차
0. 환경 준비
1. 데이터 수집 (불러오기)
2. 데이터 정제 (정규표현식)
3. 토큰화 (Kiwi 띄어쓰기 교정 + Okt 품사 태깅)
4. Word2Vec 임베딩
5. 라벨링 & 문서 벡터 만들기
6. 모델링 & 교차검증
7. 내적값으로 단어 감성 점수 구하기
8. 오늘의 결과 정리


---
## 0. 환경 준비

- `konlpy` (Okt) 는 **Java 기반** → `JAVA_HOME`을 먼저 잡아줘야 한다 (ex01에서도 똑같이 걸렸던 부분)
- GPU는 이번 실습(Word2Vec + 로지스틱 회귀)에 필수는 아니고 환경 점검용


In [93]:
# konlpy는 Java 기반이라 JDK 경로(JAVA_HOME)를 잡아줘야 동작한다
# -> 본인 PC에 설치된 JDK 경로로 수정 필요
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-25.0.4.7-hotspot"

In [94]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)   # cuda 나오면 준비 끝

cuda


---
## 1. 데이터 수집 (불러오기)

- `naver_shopping.txt` : **헤더가 없는** 탭 구분 파일 → `header=None`, `sep='\t'`
- 컬럼은 `평점(1,2,4,5)` + `리뷰(원문)` 두 개뿐 → 나중에 평점을 긍정/부정 **정답 라벨로** 사용


In [95]:
# 필요한 라이브러리 불러오기
import pandas as pd

In [96]:
# 데이터 불러오기
# header=None : 첫 줄이 컬럼명이 아니라 데이터라는 뜻 (안 쓰면 첫 리뷰가 컬럼명이 됨)
# sep='\t'   : tsv 형식이라 탭 기준으로 나눈다
data = pd.read_csv('./data/naver_shopping.txt', sep='\t', header=None)

In [97]:
# 컬럼명이 없어서 0, 1로 들어온 상태
data.head()

,0,1
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


In [98]:
# 컬럼명 수정 -> 0:평점, 1:리뷰
data.columns = ['평점', '리뷰']
data.head()

,평점,리뷰
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


---
## 2. 데이터 정제 (정규표현식)

- 리뷰 데이터라 이모티콘 · 특수문자가 섞여 있다 → **필요 없는 문자 제거**
- 20만 건이라 손으로 못 하고 **정규표현식 + 반복문**으로 컴퓨터에게 시킨다

> **포인트** : `[^...]` 는 부정(NOT) → "여기 적힌 것만 남기고 나머지는 다 지워라" 라는 뜻.
> 지울 문자를 나열하는 것보다 **남길 문자를 나열**하는 게 훨씬 편하다. (ex01에서 배운 것)


In [99]:
data.info()
# 결측치가 없는 상태 -> 20만 행 모두 Non-Null
# 평점은 int64, 리뷰는 object(문자열)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   평점      200000 non-null  int64 
 1   리뷰      200000 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.1+ MB


In [100]:
# 특수문자 제거 - 정규표현시
# 데이터의 갯수 - 20만개
# 우리가 하나씩 손보기는 데이터의 볼륨이 많다
# 그래서 컴퓨터에게 특수문자 제거하도록 시키고 싶다
# -> 일을 시키는 방법과 도구가 필요

In [101]:
# 라이브러리 불러오기
import re

In [102]:
# 특수문자를 제거하는 방식을 알려주자
# 문자의 패턴을 지정해주자
# [^ ... ] : 대괄호 안에 적힌 것을 '제외한' 나머지 -> 즉 남길 문자를 적는 것
#   a-zA-z : 영어 대소문자
#   0-9    : 숫자
#   가-힣   : 완성형 한글 (ㅋㅋ, ㅠㅠ 같은 자모는 여기 안 들어감 -> 제거됨)
#   \s      : 공백
#   \.\?\!  : 마침표, 물음표, 느낌표는 감성 표현이라 남겨둠
pattern = r'[^a-zA-z0-9가-힣\s\.\?\!]'

# ⚠️ 복습 포인트 : 'a-zA-z' 는 'a-zA-Z' 오타.
#    아스키 순서상 Z~a 사이의 [ \ ] ^ _ ` 기호까지 같이 남게 된다.
#    다음엔 r'[^a-zA-Z0-9가-힣\s\.\?\!]' 로 쓸 것

In [103]:
from tqdm.auto import tqdm  # 반복문 진행률 표시 도구 (20만 건이라 얼마나 걸릴지 확인용)

In [104]:
# 정규표현식으로 데이터 정제 해주기
# re.sub(패턴, 바꿀문자, 대상) -> 패턴에 걸린 문자를 ''(빈 문자)로 치환 = 삭제

text = data['리뷰']
new_doc = []

for doc in tqdm(text):
    cleaned_doc = re.sub(pattern, '', doc)
    new_doc.append(cleaned_doc)

  0%|          | 0/200000 [00:00<?, ?it/s]

In [105]:
# 정규표현식으로 정리한 텍스트 데이터 -> 컬럼으로 추가해보자
# (오타 주의 : cleaned_doc 인데 claened_doc 으로 만들어졌다. 아래에서 계속 이 이름을 쓴다)
data['claened_doc'] = new_doc

In [106]:
# 원문(리뷰)과 정제본(claened_doc) 비교
# -> 5번 행의 'ㅎㅎ' 가 사라진 것 확인 (자모는 가-힣에 포함 안 되므로)
data.head()

,평점,리뷰,claened_doc
0,5,배공빠르고 굿,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요


---
## 3. 토큰화 (Kiwi 띄어쓰기 교정 + Okt 품사 태깅)

리뷰는 띄어쓰기가 엉망인 경우가 많아서 **2단계**로 처리한다.

| 순서 | 처리 | 도구 | 이유 |
|---|---|---|---|
| 1 | 띄어쓰기 교정 | `kiwipiepy` 의 `kiwi.space()` | 띄어쓰기가 틀리면 형태소 분석이 통째로 망가짐 |
| 2 | 형태소 분리 + 품사 태깅 | `konlpy` 의 `Okt` | 조사·어미를 버리고 의미 있는 품사만 남김 |
| 3 | 불용어 제거 | `kiwipiepy.utils.Stopwords` | 의미 없는 단어 제거 |

> **남길 품사** : `Noun`(명사), `Verb`(동사), `Adjective`(형용사)
> → 감성은 대부분 이 3개 품사에 들어있다. 조사(Josa)·어미(Eomi)는 노이즈.

> **stem=True 의 효과** : "빠르고" → "빠르다" 처럼 **어간(기본형)으로 통일**.
> 같은 뜻인데 활용형이 달라 다른 단어로 세지는 걸 막아준다.


In [107]:
!pip install kiwipiepy


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
# 불용어(stopwords) 사전 불러오기
# 대소문자 주의 : StopWords ❌ → Stopwords ⭕
from kiwipiepy.utils import Stopwords

stopwords = Stopwords()

In [109]:
# 객체 자체를 출력하면 내용이 안 보인다 (주소값만 나옴)
stopwords

In [110]:
# 불용어 사전은 (단어, 품사) 튜플 형태 -> 단어만 뽑아내자
# 리스트 컴프리헨션 버전 (한 줄)
stopwords_list = [word for word, tag in stopwords.stopwords]

In [111]:
# 위 컴프리헨션과 '완전히 같은 코드'를 for문으로 풀어쓴 버전
# -> 컴프리헨션이 헷갈릴 때 이렇게 풀어서 이해하면 된다
stopwords_list = []

for word, tag in stopwords.stopwords:
    stopwords_list.append(word)

In [112]:
!pip install konlpy


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [113]:
# Okt : 소셜미디어/리뷰 문체에 강하고 속도가 빠른 형태소 분석기
from konlpy.tag import Okt
okt = Okt()

In [114]:
# Kiwi : 띄어쓰기 교정용으로 사용
from kiwipiepy import Kiwi
kiwi = Kiwi()

In [115]:
# 띄어쓰기 기능 테스트 해보기
# "띄어쓰기교정해" -> "띄어쓰기 교정해" 로 붙어있던 말을 띄워준다
kiwi.space("띄어쓰기교정해")

'띄어쓰기 교정해'

In [116]:
# Okt가 붙여주는 품사 태그 목록 확인
# -> 이 중에서 Noun / Verb / Adjective 만 골라 쓸 것
okt.tagset

{'Adjective': '형용사',
 'Adverb': '부사',
 'Alpha': '알파벳',
 'Conjunction': '접속사',
 'Determiner': '관형사',
 'Eomi': '어미',
 'Exclamation': '감탄사',
 'Foreign': '외국어, 한자 및 기타기호',
 'Hashtag': '트위터 해쉬태그',
 'Josa': '조사',
 'KoreanParticle': '(ex: ㅋㅋ)',
 'Noun': '명사',
 'Number': '숫자',
 'PreEomi': '선어말어미',
 'Punctuation': '구두점',
 'ScreenName': '트위터 아이디',
 'Suffix': '접미사',
 'Unknown': '미등록어',
 'Verb': '동사'}

In [117]:
# 사용자 정의 형태소 분리 함수 만들기
# 정제 -> 토큰화 과정을 하나로 묶어두면 나중에 새로운 문장에도 똑같이 적용할 수 있다
def pos_tagging(text):
    # 1. 띄어쓰기 교정 (형태소 분석 정확도를 올리기 위한 사전 작업)
    text = kiwi.space(text)

    # 2. 형태소 분리 + 품사 태깅
    pos_words = okt.pos(
        text,
        stem=True,   # 어간 추출 : '빠르고' -> '빠르다'
        norm=True,   # 정규화   : '좋아욬ㅋㅋ' -> '좋아요'
    )

    # 3. 필요한 품사만 남기고 + 불용어 제거
    tagged_list = []
    for word, tag in pos_words:
        if tag in ['Noun', 'Verb', 'Adjective']:   # 명사/동사/형용사만
            if word not in stopwords_list:          # 불용어는 버림
                tagged_list.append(word)
    return tagged_list

In [118]:
# 함수 테스트 해보기
# '배공빠르고 굿' -> ['배공', '빠르다', '굿']
# stem=True 덕분에 '빠르고'가 기본형 '빠르다'로 바뀐 것 확인
pos_tagging(data['claened_doc'].iloc[0])

['배공', '빠르다', '굿']

### ⚠️ 여기서 멈췄던 부분

20만 건 × (띄어쓰기 교정 + 형태소 분석) 은 시간이 아주 오래 걸린다 → 중간에 중단(KeyboardInterrupt).

그래서 아래 셀에서 `data['tagged_doc'] = tagged_doc` 이 **길이 불일치 에러**가 났다.
- `tagged_doc` : 중단된 시점까지 2,780개
- `data` : 200,000행
- 👉 **DataFrame에 리스트를 컬럼으로 넣으려면 길이가 정확히 같아야 한다**

**해결** : 미리 토큰화가 끝난 `naver_shopping(토큰화완료).pkl` 을 불러와서 이어서 진행.
(실무에서도 오래 걸리는 전처리는 **한 번만 돌리고 파일로 저장**해두는 게 정석)


In [119]:
# 전체 데이터 함수 적용시켜보기
tagged_doc = []

for text in tqdm(data['claened_doc']):
    tagged_doc.append(pos_tagging(text))
    

  0%|          | 0/200000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 데이터 프레임에 추가
data['tagged_doc'] = tagged_doc

ValueError: Length of values (2780) does not match length of index (200000)

In [ ]:
# 전처리 데이터 파일 저장하기
import pickle

with open('./data/naver_shopping(토큰화완료).pkl', 'wb') as f:
    pickle.dump(data, f)

---
## 4. Word2Vec 임베딩

### 왜 임베딩이 필요한가
- 컴퓨터는 문자열을 못 읽는다 → **숫자 벡터**로 바꿔야 한다
- TF-IDF(ex01)는 "얼마나 자주 나오냐"만 셌기 때문에 **단어의 의미는 모른다**
  (`좋다` 와 `훌륭하다` 가 완전히 남남)
- Word2Vec은 **주변에 같이 등장하는 단어가 비슷하면 벡터도 비슷해지도록** 학습
  → 벡터 간 거리로 **유사도 계산**이 가능해진다

### 학습 방식 2가지
| 방식 | 설명 |
|---|---|
| **CBOW** | 주변 단어들로 → 가운데 단어를 맞춘다 (빠름) |
| **Skip-gram** | 가운데 단어로 → 주변 단어들을 맞춘다 (희귀 단어에 강함, `sg=1`) |

> 이번 실습은 **Skip-gram (`sg=1`)** 사용


In [120]:
# 1. 임베딩 도구 word2vec(gensim) 설치
import sys
!{sys.executable} -m pip install -q --upgrade gensim

# 2. 한글 폰트는 윈도우에 이미 있으니 설치 대신 matplotlib에 등록만
#    (등록 안 하면 그래프의 한글이 네모(□)로 깨진다)
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 커널 재시작 지점
> 앞의 전처리를 다시 돌릴 필요 없이, **여기서부터 실행**하면 된다.


In [121]:
import pandas as pd
import pickle
from tqdm.auto import tqdm

In [122]:
# 토큰화까지 끝난 데이터 불러오기 (pickle : 파이썬 객체를 그대로 저장/복원)
# 'rb' = read binary
with open('./data/naver_shopping(토큰화완료).pkl', 'rb') as f:
    data = pickle.load(f)

In [123]:
# 평점 / 리뷰 / claened_doc / tagged_doc 4개 컬럼, 20만 행 확인
data

,평점,리뷰,claened_doc,tagged_doc
0,5,배공빠르고 굿,배공빠르고 굿,"[배공, 빠르다, 굿]"
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,"[택배, 엉망, 용, 저희, 집, 밑, 층, 놔두다, 가다]"
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,"[아주, 좋다, 바지, 정말, 좋다, 개, 구매, 하다, 가격, 대박, 이다, 바느..."
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,"[선물, 용, 받다, 전달, 하다, 하다, 상품, 이다, 머그, 컵, 오다, 당황,..."
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요,"[민트, 색상, 예쁘다, 옆, 손잡이, 거, 용, 도로, 사용, 되다]"
...,...,...,...,...
199995,2,장마라그런가!!! 달지않아요,장마라그런가!!! 달지않아요,"[장마, 런가, 달, 않다]"
199996,5,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...,"[다이슨, 케이스, 구매, 하다, 다이슨, 슈퍼, 소닉, 드라이기, 케이스, 구매,..."
199997,5,로드샾에서 사는것보다 세배 저렴하네요 ㅜㅜ 자주이용할께요,로드샾에서 사는것보다 세배 저렴하네요 자주이용할께요,"[로드샾, 살다, 세, 배, 저렴하다, 자주, 이용, 하다]"
199998,5,넘이쁘고 쎄련되보이네요~,넘이쁘고 쎄련되보이네요,"[넘다, 이쁘다, 쎄다, 되다, 보이다]"


### 4-1. 사전학습(pre-trained) 모델 먼저 체험해보기

직접 학습하기 전에, 구글 뉴스로 학습된 모델로 **"임베딩이 의미를 담는다"** 를 눈으로 확인한다.


In [ ]:
# 임베딩을 하는 이유
# 컴퓨터는 단어를 이해하지 못한다 -> 수치값으로 바꿔줘야한다
# 컴퓨터가 이해할 수 있는 숫자된 값을 치환해주는 방법이 임베딩
# 단어가 가진 의미를 녹여내서 실수의 형태로 치환
# 임베딩이 완료되면 단어의 의미가 수치로 변환되기 때문에 유사도 분석이 가능

In [125]:
# gensim이 제공하는 사전학습 모델 다운로드 도구
import gensim.downloader as api

- Records : 레코드의 수, 즉 데이터의 개수를 의미(각 "레코드"는 임베딩된 단어 하나를 의미)

| 모델 이름                           | 레코드 수      | 설명                                                                                                  |
|-------------------------------------|----------------|-------------------------------------------------------------------------------------------------------|
| conceptnet-numberbatch-17-06-300    | 1,917,247      | ConceptNet Numberbatch는 개념 간의 관계를 학습한 모델.                                                 |
|                                     |                | 상식을 이해하고 개념 간 관계를 연결시키기 위해 개발됨.                                                 |
|                                     |                | 주로 의미 네트워크 연구에 사용됨. "고양이"와 "동물" 간의 관계를 학습.                                  |
| fasttext-wiki-news-subwords-300     | 999,999        | fastText는 단어뿐만 아니라 서브워드(단어의 작은 부분)를 학습.                                          |
|                                     |                | "unbelievable"을 "un", "believe", "able"로 나누어 학습.                                                 |
|                                     |                | Wikipedia와 뉴스 기사 데이터를 기반으로 학습됨.                                                       |
| glove-twitter-100                   | 1,193,514      | GloVe는 Stanford에서 개발된 모델로, 단어 간의 공간적 관계를 벡터로 표현.                               |
|                                     |                | 트위터 데이터(2억 개 트윗)로 학습되어 소셜 미디어 자연어 처리 작업에 적합.                             |
| word2vec-google-news-300            | 3,000,000      | Word2Vec은 구글 뉴스 데이터로 학습된 모델.                                                             |
|                                     |                | 뉴스 기사의 단어 간 유사성 및 의미적 관계를 잘 포착.                                                   |
|                                     |                | 뉴스 데이터 기반이기 때문에 자연어 처리에 많이 사용됨.                                                 |
| word2vec-ruscorpora-300             | 184,973        | Word2Vec ruscorpora는 러시아어 텍스트 기반의 모델로,                                                   |
|                                     |                | 러시아어 자연어 처리 작업에 사용됨.                                                                   |


In [126]:
# 여러 모델 중에 word2vec-google-news-300를 사용
# (300차원 / 300만 단어. 용량이 커서 처음 실행 시 다운로드에 시간이 걸린다)
model = api.load('word2vec-google-news-300')

In [127]:
# 단어 벡터 연산 : soju - korea + mexico = ?
# "한국의 소주에 해당하는 멕시코의 술은?" 이라는 뜻
# -> tequila (데킬라) 가 1등! 벡터 연산만으로 의미 관계를 찾아낸다
print(model.most_similar_cosmul(positive=['soju', 'mexico'], negative=['korea']))

[('tequila', 0.8992794156074524), ('mezcal', 0.8555493950843811), ('agave_tequila', 0.8524277806282043), ('Modelo_Especial', 0.836313784122467), ('pulque', 0.8301872611045837), ('mescal', 0.8242558240890503), ('distilled_liquor', 0.8173635601997375), ('Agavero', 0.8148321509361267), ('rum', 0.8130227327346802), ('michelada', 0.8111985325813293)]


In [128]:
# 같은 방식으로 러시아 버전 -> vodka (보드카)
# 임베딩 벡터가 '술'이라는 의미와 '나라'라는 의미를 따로 담고 있다는 증거
print(model.most_similar_cosmul(positive=['soju', 'russia'], negative=['korea']))

[('vodka', 0.8616750240325928), ('brandy', 0.8266340494155884), ('distilled_liquor', 0.8266003727912903), ('Ochakovo', 0.8215005397796631), ('Campari', 0.8179371953010559), ('brandy_cognac', 0.8096943497657776), ('Bombay_Sapphire_gin', 0.8087176084518433), ('plum_brandy', 0.8080827593803406), ('Spanish_cava', 0.8054169416427612), ('whiskey_brandy', 0.8039817214012146)]


### 4-2. 우리 데이터로 직접 Word2Vec 학습하기

사전학습 모델은 **영어 뉴스** 기반이라 한국어 쇼핑 리뷰에는 못 쓴다.
→ 우리가 만든 `tagged_doc` 으로 **직접 학습**한다.

| 파라미터 | 값 | 의미 |
|---|---|---|
| `sentences` | `data['tagged_doc']` | 학습 데이터 (토큰 리스트의 리스트) |
| `window` | 3 | 앞뒤 3단어까지를 '주변 단어'로 본다 |
| `min_count` | 5 | 5번 미만 등장한 단어는 학습에서 제외 (오타·희귀어 걸러내기) |
| `sg` | 1 | 1=Skip-gram, 0=CBOW |
| `vector_size` | 100 | 단어 하나를 100개의 숫자로 표현 |
| `negative` | 5 | 네거티브 샘플링 개수 |


In [129]:
# 우리가 가진 데이터를 가지고 학습을 시켜서 사용
from gensim.models import Word2Vec

In [130]:
# 모델 생성 및 학습 진행 (생성과 동시에 학습까지 진행된다)
w2v = Word2Vec(
    window = 3,          # 한 단어와 관련된 인접한 단어의 범위를 지정
    min_count = 5,       # 특정 단어가 학습에 포함되기 위한 최소 등장 횟수
    sg = 1,              # 1 = Skip-gram (가운데 단어로 주변 단어를 예측)
    vector_size = 100,   # 단어 1개를 100차원 벡터로 표현
    negative = 5,        # 네거티브 샘플링 개수
    sentences = data['tagged_doc']   # 학습 데이터
)
# 네거티브 샘플링 : 정답 단어만 학습하면 계산량이 너무 크니까,
#                  관계없는 단어를 무작위로 몇 개 뽑아 '오답'으로 같이 학습시켜 속도를 올리는 기법

In [131]:
# '배송' 이라는 단어가 어떤 숫자로 바뀌었는지 확인
# -> 100개의 실수값. 사람은 못 읽지만 컴퓨터는 이걸로 의미를 계산한다
w2v.wv.get_vector('배송')

array([-0.2775932 , -0.0916969 ,  0.11946485, -0.7046477 , -0.03485557,
       -0.02730699, -0.0107522 ,  0.4954271 , -0.6025905 , -0.02389004,
       -0.17806783, -0.15013249, -0.24335839, -0.33464134,  0.08630874,
       -0.17662634, -0.08704524,  0.02602367, -0.18527111, -0.34162176,
       -0.5681212 , -0.11342536,  0.35992873,  0.01841948, -0.36148846,
        0.01902116,  0.07813314, -0.14925295,  0.02885758,  0.0081108 ,
        0.41540974, -0.14559743,  0.08432356, -0.8772128 ,  0.14131902,
        0.6590727 ,  0.2207326 , -0.12269016, -0.09327721, -0.30352882,
       -0.06860875, -0.41446403, -0.4312037 ,  0.20365664,  0.14230478,
       -0.14966835, -0.2284463 , -0.0492379 ,  0.15679252,  0.0871029 ,
        0.57171315, -0.2691466 ,  0.14984713,  0.2669451 , -0.0457767 ,
        0.3779798 ,  0.4451011 , -0.3092862 , -0.4202092 , -0.32491383,
        0.42302954, -0.14385384,  0.31166562, -0.653837  , -0.02072892,
        0.2684902 ,  0.2590577 ,  0.40494826, -0.6302785 ,  0.14

In [132]:
# '만족' 과 가장 비슷한 단어 20개
# -> 만족하다 / 만족스럽다 / 최상 / 훌륭하다 ... 의미가 비슷한 단어가 잘 묶였다
# 다만 '합닏', '젛', '괞찬' 같은 오타·비표준어도 섞여 있다
#   -> 리뷰 데이터 특성상 오타가 비슷한 자리에 등장하기 때문 (정제의 한계)
w2v.wv.most_similar('만족', topn=20)

[('만족하다', 0.8381425738334656),
 ('만족스럽다', 0.7545600533485413),
 ('최상', 0.7313949465751648),
 ('훌륭하다', 0.708719789981842),
 ('쿠', 0.6946830749511719),
 ('맘에듬', 0.6908121705055237),
 ('흐뭇하다', 0.6776658296585083),
 ('만점', 0.6757915019989014),
 ('합닏', 0.6672874093055725),
 ('젛', 0.6671707034111023),
 ('민족', 0.6647335290908813),
 ('대대', 0.6613532900810242),
 ('핮니', 0.652235746383667),
 ('합리', 0.6485459804534912),
 ('퀄러티', 0.6424337029457092),
 ('최강', 0.6402283310890198),
 ('쫀쫀합니', 0.6333039402961731),
 ('령', 0.6311812400817871),
 ('괞찬', 0.6301233172416687),
 ('힙니', 0.630113959312439)]

---
## 5. 라벨링 & 문서 벡터 만들기

### 5-1. 평점 → 긍정/부정 라벨
- 이 데이터는 **평점 3점이 없다** (1, 2, 4, 5점만 존재) → 중립을 뺀 깔끔한 이진 분류가 가능
- `4점 이상 = 긍정(1)` / `2점 이하 = 부정(0)`


In [133]:
# 평점 분포 확인 -> 3점이 없다 (1,2,4,5 만 존재)
data['평점'].value_counts()

평점
5    81177
2    63989
1    36048
4    18786
Name: count, dtype: int64

In [134]:
# 평점 기반 4점과 5점은 긍정(1) / 나머지는 부정(0)으로 변환
label = []

for rating in data['평점']:
    if rating > 3:
        label.append(1)   # 긍정
    else:
        label.append(0)   # 부정

In [135]:
data['label'] = label

In [136]:
# label 컬럼 추가 확인
data.head()

,평점,리뷰,claened_doc,tagged_doc,label
0,5,배공빠르고 굿,배공빠르고 굿,"[배공, 빠르다, 굿]",1
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,"[택배, 엉망, 용, 저희, 집, 밑, 층, 놔두다, 가다]",0
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,"[아주, 좋다, 바지, 정말, 좋다, 개, 구매, 하다, 가격, 대박, 이다, 바느...",1
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,"[선물, 용, 받다, 전달, 하다, 하다, 상품, 이다, 머그, 컵, 오다, 당황,...",0
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요,"[민트, 색상, 예쁘다, 옆, 손잡이, 거, 용, 도로, 사용, 되다]",1


### 5-2. 단어 벡터 → 문서 벡터 (평균 벡터)

**핵심 문제** : Word2Vec은 **단어 1개 = 벡터 1개**를 준다. 그런데 우리가 분류할 대상은 **리뷰(문장)** 다.

**해결** : 리뷰에 들어있는 모든 단어 벡터를 구해서 **평균**을 낸다 → 리뷰 1개 = 100차원 벡터 1개

```
'배송 빠르다 좋다'
  ↓ 각 단어를 100차원 벡터로
[0.1, -0.3, ...]   ← 배송
[0.2,  0.5, ...]   ← 빠르다
[0.4,  0.1, ...]   ← 좋다
  ↓ 각 차원(열)별로 평균 (axis=0)
[0.23, 0.1, ...]   ← 이 리뷰 1개를 대표하는 벡터
```

> **주의** : `min_count=5` 때문에 모든 토큰이 벡터를 가진 건 아니다.
> 남은 토큰이 하나도 없는 리뷰는 `np.zeros(100)` (영벡터)로 채워서 **행 개수를 20만으로 유지**한다.


In [137]:
import numpy as np

X_w2v_list = []

# 학습된 모델에서 벡터 차원 수를 꺼내온다 (Word2Vec에 넘긴 vector_size=100 과 동일)
vector_size = w2v.wv.vector_size

# 평균 벡터 계산해보기
for doc in data["tagged_doc"] :
    vecs = [] # 해당 리뷰의 각 벡터값들을 담아둘 리스트

    for token in doc : # 각 문장의 토큰에 대해 순회
        if token in w2v.wv : # Word2Vec에 해당 토큰이 들어있니? (min_count=5로 걸러진 단어는 없음)
            vecs.append(w2v.wv[token]) # 벡터 추가

    if len(vecs) > 0: # 벡터들의 길이값이 0보다 큰가? -> 정제하는 과정에서 없어질 수도 있음!
        avg_vec = np.mean(vecs, axis = 0) # axis=0 : 각 차원(열)별로 평균 계산
    else : # 벡터가 하나도 없는 경우 -> 0으로 채운다 (행 개수를 맞춰주기 위해!)
        avg_vec = np.zeros(vector_size)

    X_w2v_list.append(avg_vec)

In [138]:
# 원본 데이터와 같은 20만 개인지 확인 (여기서 개수가 안 맞으면 모델링 단계에서 에러)
len(X_w2v_list)

200000

In [139]:
# 모델링을 위해서 2차원 배열 형태로 변환
# vstack : (100,) 짜리 벡터 20만 개를 세로로 쌓아서 (200000, 100) 배열로 만든다
X_w2v = np.vstack(X_w2v_list)

---
## 6. 모델링 & 교차검증

- 모델 : **로지스틱 회귀** (ex01과 동일. 빠르고 해석이 쉬운 베이스라인)
- 평가 : **StratifiedKFold 5겹 교차검증**
  - `Stratified` = 각 폴드마다 **긍정/부정 비율을 원본과 똑같이** 유지
  - train/test 한 번만 나누는 것보다 **성능 추정이 안정적**


In [140]:
# 데이터 전처리 완료 -> 모델링 시작
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [142]:
cv = StratifiedKFold(
    n_splits = 5,        # 데이터를 5등분해서 5번 학습/평가
    shuffle = True,      # 나누기 전에 섞기 (데이터 순서에 편향되지 않게)
    random_state = 2026  # 섞는 방식 고정 -> 재실행해도 같은 결과
)

logi = LogisticRegression(
    max_iter = 10000,    # 반복 횟수 (기본 100은 부족해서 수렴 경고가 뜬다)
    random_state = 2026
)

# cross_val_score : 5번 학습 + 5번 평가를 알아서 해준다
scores = cross_val_score(
    logi,
    X_w2v,           # 문제 : 리뷰 평균 벡터 (200000, 100)
    data['label'],   # 정답 : 0(부정) / 1(긍정)
    cv = cv
)

In [144]:
# 5개 폴드 점수의 평균 = 약 0.8736 (87.4%)
# TF-IDF 없이 100차원만으로 이 정도면 임베딩이 의미를 잘 담았다는 뜻
scores.mean()

np.float64(0.8735799999999999)

In [145]:
# 교차검증은 '평가'만 한 것이라 logi 객체에는 학습 결과가 남아있지 않다
# -> 아래 단계(계수 활용)를 위해 전체 데이터로 한 번 더 학습시켜준다
print(logi.fit(X_w2v, data['label']))

LogisticRegression(max_iter=10000, random_state=2026)


---
## 7. 내적값으로 단어 감성 점수 구하기 ⭐ (오늘의 하이라이트)

### 원리
로지스틱 회귀는 학습이 끝나면 **계수(coefficient) 벡터**를 갖는다. 여기서는 100차원.
이 계수 벡터는 "**어느 방향이 긍정인가**"를 가리키는 화살표다.

그러면 **단어 벡터 · 계수 벡터의 내적(dot product)** 은
"이 단어가 긍정 방향을 얼마나 향하고 있나" = **그 단어의 감성 점수**가 된다.

```
내적 = (a1 × b1) + (a2 × b2) + ... + (a100 × b100)
```
| 내적값 | 의미 |
|---|---|
| **양수 (+)** | 긍정 단어 |
| **음수 (−)** | 부정 단어 |
| 0에 가까움 | 중립 |

> 문장 단위 예측만 하는 게 아니라 **단어 하나하나의 감성까지 뜯어볼 수 있다**는 게 핵심.
> → 감성 사전을 사람이 안 만들고 **데이터에서 자동으로 뽑아낼 수 있다**


In [147]:
# 데이터 내에 등장하는 단어 집합 생성 (w2v에 존재하는 단어만 고려)
# set : 중복 자동 제거
word_set = set()
for tokens in data['tagged_doc']:
    for token in tokens:
        if token in w2v.wv:
            word_set.add(token)

# word_set 집합 : 모델에서 임베딩된 단어들만 포함한 집합
# -> 여기 없는 단어를 w2v.wv[word] 로 조회하면 KeyError가 난다

In [ ]:
# 긍정적인 단어와 부정적인 단어를 기반으로 테스트를 진행
# Dot_product : 두 벡터를 곱하는 연산을 진행
# 두 벡터의 같은 인덱스에 위치한 원소들을 곱한 후 그 결과를 모두 더한 값
# 내적값이 양수(+) : 긍정 감성에 해당하는 값이다.
# 내적값이 음수(-) : 부정 감성에 해당하는 값이다.

In [149]:
# 긍정적인 단어와 부정적인 단어 기반으로 테스트

# 실제 데이터에 포함된 단어 사용해야 함 (word_set 안에 있는 단어여야 KeyError가 안 남)
sample_words = ["깔끔하다", "실망"]

coef_vector = logi.coef_[0]  # 로지스틱 회귀 모델의 계수 (100차원)
coef_vector.shape            # vector_size 와 동일한 (100,)

for word in sample_words:
    # 1. 단어의 임베딩 벡터 추출
    embedding_vector = w2v.wv[word]

    # 2. 모델 계수 벡터와의 내적을 np.dot()을 이용하여 계산
    dot_product = np.dot(embedding_vector, coef_vector)

    # 3. 결과 출력
    #    깔끔하다 -> +10.39 (긍정) / 실망 -> -13.49 (부정)  기대한 대로 부호가 갈렸다!
    print(f"단어 : {word}")
    print("np.dot() 결과:", dot_product)
    print("-" * 80)

단어 : 깔끔하다
np.dot() 결과: 10.394273408814836
--------------------------------------------------------------------------------
단어 : 실망
np.dot() 결과: -13.488439044403902
--------------------------------------------------------------------------------


---
## 8. 오늘의 결과 정리

### 결과
- 교차검증 정확도 **약 0.874 (87.4%)** — 100차원 평균 벡터만으로 얻은 성능
- 단어 감성 점수 : `깔끔하다` **+10.39** / `실망` **−13.49** → 부호가 기대대로 갈렸다
- Word2Vec 유사도 : `만족` → 만족하다 / 만족스럽다 / 최상 / 훌륭하다 (의미가 잘 묶임)

### 배운 것
1. **임베딩** : TF-IDF는 빈도만 세지만, Word2Vec은 **단어의 의미**를 벡터에 담는다
   → 그래서 유사도 계산·단어 연산(`soju - korea + mexico = tequila`)이 가능해진다
2. **CBOW vs Skip-gram** : 주변→가운데(CBOW) / 가운데→주변(Skip-gram, `sg=1`)
3. **단어 벡터 → 문서 벡터** : 리뷰 안의 단어 벡터들을 **평균(axis=0)** 내면 문서 하나를 대표하는 벡터가 된다
   - 벡터가 하나도 없는 리뷰는 **영벡터**로 채워 행 개수를 맞춰야 한다
4. **내적(dot product)의 의미** : 단어 벡터 · 모델 계수 = 그 단어의 **감성 점수** (양수=긍정, 음수=부정)
5. **StratifiedKFold** : 클래스 비율을 유지한 채 5겹 교차검증 → 한 번 분할보다 신뢰도가 높다
6. **교차검증은 학습된 모델을 남기지 않는다** → 계수를 쓰려면 `fit()` 을 한 번 더 해야 한다
7. **오래 걸리는 전처리는 파일로 저장** : 20만 건 형태소 분석은 중간에 끊기면 처음부터.
   `pickle`로 저장해두고 다음부터는 불러쓰기

### 오늘 막혔던 부분 (다시 안 틀리기)
| 문제 | 원인 | 해결 |
|---|---|---|
| `Length of values (2780) does not match length of index (200000)` | 형태소 분석이 중간에 중단됨 | 미리 토큰화된 pkl 로드 |
| `Stopwords` import 실패 | `StopWords` 대소문자 오타 | `Stopwords` |
| 패턴에 이상한 기호가 남음 | `a-zA-z` (Z가 소문자 z) 오타 | `a-zA-Z` |
| 컬럼명 `claened_doc` | `cleaned` 오타 | 다음엔 이름부터 확인 |

### 다음에 시도해볼 것
- 평균 벡터 대신 **TF-IDF 가중 평균** 또는 **Doc2Vec** 으로 문서 벡터 만들어보기
- `word_set` 전체를 내적으로 계산해서 **긍정/부정 단어 TOP 20 랭킹** 뽑아보기
- Word2Vec 파라미터 튜닝 (`window`, `vector_size`, `sg=0` CBOW 비교)
- `classification_report` / confusion matrix 로 precision·recall 확인
- 오타·비표준어(`합닏`, `괞찬`) 정제 강화 → 유사도 품질 개선
- TF-IDF(ex01) vs Word2Vec(ex02) **성능 직접 비교**
